In [1]:
import numpy as np
from statistics import mean
import random

SAMPLE_SIZE = 1000
SEED = 3210

In [2]:
random.seed(SEED)
np.random.seed(SEED)

### Numerical excercise

To cast the story in a realistic setting, we imagine the testing of a new drug, aimed to help patients
suffering from a deadly disease. An RCT is conducted to evaluate the efficacy of the drug and it is found
to be 28% effective in both males and females; in other words, CATE(male) = CATE(female) = 0.28. The
drug is approved and, after a year of use, a follow-up randomized study is conducted yielding the same
results; namely CATE remained 0.28 and, moreover, men and women remained totally indistinguishable
in their responses, as shown in Table 1

(Scott Mueller and Judea Pearl, 2023, p.4)

In [3]:
# Table 1 (RCT data)

table_1 = np.array([   # Female    |   Male
                    [489/1_000, 490/1_000], # do(drug)
                    [210/1_000, 210/1_000], # do(no drug)
])

CATE = table_1[0] - table_1[1]
ATE = mean(CATE)

In [4]:
print(CATE)
print(ATE)

[0.279 0.28 ]
0.2795


In [5]:
table_1

array([[0.489, 0.49 ],
       [0.21 , 0.21 ]])


At this point, the drug manufacturer ventured to find out to what degree people actually buy the
approved drug, following its recommended usage. A market survey was conducted (observational study)
and revealed that only 70% of men and 70% of women actually chose to take the drug; problems with side
effects and rumors of unexpected deaths may have caused the other 30% to avoid it. A careful examination
of the observational study has further revealed substantial differences in survival rates of men and women
who chose to use the drug (shown in Tables 2 and 3). The rate of recovery among drug-choosing men was
exactly the same as that among the drug-avoiding men (70% for each), but the rate of recovery among
drug-choosing women was 43% lower than among drug-avoiding women (0.27 vs 0.70, in Table 2). It appears
as though many women who chose the drug were already in an advanced stage of the disease, which may
account for their low recovery rate of 27%.

(Scott Mueller and Judea Pearl, 2023, p.4)

In [6]:
exp_data = np.concat([table_1*1_000, [[1_000], [1_000]]], axis=1)

In [7]:
exp_data

array([[ 489.,  490., 1000.],
       [ 210.,  210., 1000.]])

In [8]:
obs_data = np.array([ # Female | Male | Total
                        [378,   980,    1400], # drug
                        [420,   420,    600], # no drug
])

# NOTE The total is not a sum of both sexes, it is the total for each sex.

In [9]:
# from an isolated perspective, it could be thought that the drug is indiferent
# to men, and that it is dangerous for women.

obs_data[:, 0:2]/obs_data[:, 2:3]

array([[0.27, 0.7 ],
       [0.7 , 0.7 ]])

In [10]:
# yt: recovery among RCT treatment group
y_do_t = table_1[0]

# yc: recovery among RCT control group
y_do_c = table_1[1]

# P(y|t, Gender) recovery among drug-choosers
y_given_t = obs_data[0, 0:2]/obs_data[0, 2:3]

# P(y|c, Gender) recovery among drug-avoiders
y_given_c = obs_data[1, 0:2]/obs_data[1, 2:3]

# P(t) probability of taking the drug
p_t = np.repeat(obs_data[0, 2]/obs_data[:, 2].sum(), 2)

p_y = y_given_t*p_t + y_given_c*(1-p_t)

print("            female | male")
print(f"y_do_t    =", y_do_t)
print(f"y_do_c    =", y_do_c)
print(f"y_given_t =", y_given_t)
print(f"y_given_c =", y_given_c)
print(f"p_t       =", p_t)
print(f"p_y       =", p_y)

            female | male
y_do_t    = [0.489 0.49 ]
y_do_c    = [0.21 0.21]
y_given_t = [0.27 0.7 ]
y_given_c = [0.7 0.7]
p_t       = [0.7 0.7]
p_y       = [0.399 0.7  ]


In [11]:
# P(benefit) = P(yt, y'c)
# The probability that an individual would both recover if assigned to the
# treatment group and sie if assigned to control.

# Consistency: yt = y|t and yc = y|c
print([
    np.repeat(0, 2),
    y_do_t - y_do_c,
    p_y - y_do_c,
    y_do_t - p_y
])

lower_bond = np.max([
    np.repeat(0, 2),
    y_do_t - y_do_c,
    p_y - y_do_c,
    y_do_t - p_y
], axis=0)

[array([0, 0]), array([0.279, 0.28 ]), array([0.189, 0.49 ]), array([ 0.09, -0.21])]


In [12]:
print([
    y_do_t,
    1 - y_do_c,
    p_t*y_given_t + (1-p_t)*(1-y_given_c),
    y_do_t - y_do_c + (p_t*(1-y_given_t)) + ((1-p_t)*y_given_c)
])

upper_bond = np.min([
    y_do_t,
    1 - y_do_c,
    p_t*y_given_t + (1-p_t)*(1-y_given_c),
    y_do_t - y_do_c + (p_t*(1-y_given_t)) + ((1-p_t)*y_given_c)
], axis=0)

[array([0.489, 0.49 ]), array([0.79, 0.79]), array([0.279, 0.58 ]), array([1. , 0.7])]


In [13]:
# What's behind this? Observational data incorporates individuals whims.

In [14]:
if bool((upper_bond == lower_bond).all()):
    print(
        f"Probability of benefit for female people is "
        f"{lower_bond[0]:.2f}\n"
        f"Probability of benefit for male people is "
        f"{lower_bond[1]:.2f}"
    )
else:
    print(
        f"Probability of benefit for female people is "
        f"between {lower_bond[0]:.2f} and {upper_bond[0]:.2f}\n"
        f"Probability of benefit for male people is "
        f"between {lower_bond[1]:.2f} and {upper_bond[1]:.2f}"
    )

Probability of benefit for female people is 0.28
Probability of benefit for male people is 0.49


In [ ]:
# probability of harm
# A property called monotonicity means that p(harm) = 0.
# We are not sure in advance, though, whether it holds for current data.

# Given the following equality: P(harm) = P(benefit) - ATE,
# we can compute the probability of harm as follows:

# p(harm|gender) = p(benefit|gender) - CATE(gender)

lower_harm = lower_bond - CATE
upper_harm = upper_bond - CATE

if bool((upper_harm == lower_harm).all()):
    print(
        f"Probability of harm for female people is "
        f"{lower_harm[0]:.2f}\n"
        f"Probability of harm for male people is "
        f"{lower_harm[1]:.2f}"
    )
else:
    print(
        f"Probability of harm for female people is "
        f"between {lower_harm[0]:.2f} and {upper_harm[0]:.2f}\n"
        f"Probability of harm for male people is "
        f"between {lower_harm[1]:.2f} and {upper_harm[1]:.2f}"
    )
# After computing the probability of harm, we can see that for female people monotonicity holds.

Probability of harm for female people is 0.00
Probability of harm for male people is 0.21
